# Workshop Part 3: Storage Theory (MLE - Machine Learning Engineer)

## Learning Objectives
- Understand data storage options and when to use them
- Learn about cloud storage (Azure Blob Storage)
- Understand data lifecycle management
- See real-world example: AH Dynamic Markdown

## Where to Store Data?

This section covers the **theoretical** aspects of data storage. We'll discuss different storage options, when to use them, and how they fit into the data lifecycle.

In [1]:
# Workshop Progress Tracker
notebooks = ["01 Extract", "02 Prepare", "03 Storage", "04 ML", "05 Deploy"]
current = 2  # This is notebook 03

print("="*70)
print("📊 WORKSHOP PROGRESS")
print("="*70)
for i, nb in enumerate(notebooks):
    if i < current:
        print(f"✅ {nb}")
    elif i == current:
        print(f"👉 {nb} ← YOU ARE HERE")
    else:
        print(f"⬜ {nb}")
print("="*70)

📊 WORKSHOP PROGRESS
✅ 01 Extract
✅ 02 Prepare
👉 03 Storage ← YOU ARE HERE
⬜ 04 ML
⬜ 05 Deploy


## 1. Data Lifecycle Stages

Data goes through multiple stages in its lifecycle:

```
Raw Data → Cleaned Data → Model Outputs → Analytics/Reports
   ↓           ↓              ↓               ↓
Extract    Transform        Load          Analyze
```

Each stage requires different storage considerations:

### Raw Data (Bronze Layer)
- **What**: Original, unprocessed data from sources
- **Example**: `product_reviews.csv` (as extracted from Kaggle)
- **Storage**: Cheap, durable storage (e.g., Azure Blob Storage, AWS S3)
- **Retention**: Keep for compliance, debugging, reprocessing

### Cleaned Data (Silver Layer)
- **What**: Processed, validated, deduplicated data
- **Example**: `cleaned_reviews.csv` (after data preparation)
- **Storage**: Fast access storage for analysis and ML
- **Retention**: Keep while actively used, can be regenerated from raw data

### Model Outputs (Gold Layer)
- **What**: ML predictions, aggregations, business metrics
- **Example**: Sentiment scores, discount recommendations
- **Storage**: Database or data warehouse for quick queries
- **Retention**: Keep for auditing, reporting, A/B testing

### Analytics/Reports
- **What**: Dashboards, visualizations, summary reports
- **Example**: PowerBI dashboards showing product sentiment trends
- **Storage**: BI tools, APIs, web applications
- **Retention**: Real-time or near real-time access

## 2. Storage Options

### A. Local File Storage (What We're Using)
**Format**: CSV, JSON, Parquet

**Pros**:
- Simple and easy to use
- No setup required
- Fast for small datasets
- Good for prototyping

**Cons**:
- Not scalable for large datasets
- No collaboration features
- No versioning
- Risk of data loss

**When to use**: Development, testing, small projects, workshops

In [2]:
import pandas as pd
import os

# Example: Check our local storage
data_path = '../data/'

print("Local Storage Structure:")
print("=" * 60)
for root, dirs, files in os.walk(data_path):
    level = root.replace(data_path, '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        file_size = os.path.getsize(file_path) / 1024  # KB
        print(f"{subindent}{file} ({file_size:.2f} KB)")

Local Storage Structure:
/
cleaned/
  .gitkeep (0.00 KB)
  cleaned_reviews.csv (99.17 KB)
raw/
  reviews.csv (51.69 KB)
  .gitkeep (0.00 KB)
  inventory.csv (10.31 KB)
  sales.csv (10.16 KB)


### B. Cloud Storage (Azure Blob Storage)
**What Albert Heijn Uses for Dynamic Markdown**

**Pros**:
- Scalable (petabytes of data)
- Durable (99.999999999% durability)
- Cost-effective for large data
- Accessible from anywhere
- Built-in versioning and lifecycle management

**Cons**:
- Network latency for access
- Requires cloud infrastructure
- Cost for egress (data transfer out)

**When to use**: Production systems, large datasets, team collaboration

#### Azure Blob Storage Tiers

| Tier | Use Case | Access Time | Cost |
|------|----------|-------------|------|
| **Hot** | Frequently accessed data (e.g., active datasets, model inputs) | Immediate | $$ |
| **Cool** | Infrequently accessed (e.g., older data, backups) | Immediate | $ |
| **Archive** | Rarely accessed (e.g., compliance, long-term retention) | Hours | ¢ |

#### Example: Accessing Azure Blob Storage (Code Reference)
```python
from azure.storage.blob import BlobServiceClient

# Connect to Azure Storage
connection_string = "your_connection_string"
blob_service_client = BlobServiceClient.from_connection_string(connection_string)

# Download a file from blob storage
container_client = blob_service_client.get_container_client("product-reviews")
blob_client = container_client.get_blob_client("raw/product_reviews.csv")

with open("local_file.csv", "wb") as download_file:
    download_file.write(blob_client.download_blob().readall())

# Upload cleaned data back to blob storage
upload_blob_client = container_client.get_blob_client("cleaned/cleaned_reviews.csv")
with open("../data/cleaned/cleaned_reviews.csv", "rb") as data:
    upload_blob_client.upload_blob(data, overwrite=True)
```

### C. Databases

#### SQL Databases (Structured)
**Examples**: PostgreSQL, MySQL, Azure SQL Database

**Pros**:
- ACID transactions (data integrity)
- Powerful queries (joins, aggregations)
- Schema enforcement
- Good for relational data

**Cons**:
- Harder to scale horizontally
- Schema changes can be complex
- Not ideal for unstructured data

**When to use**: Transactional data, relational data, real-time queries

#### NoSQL Databases (Semi-structured)
**Examples**: MongoDB, Cosmos DB, DynamoDB

**Pros**:
- Flexible schema
- Horizontally scalable
- Fast for specific query patterns
- Good for JSON-like documents

**Cons**:
- Limited joins and transactions
- May require more storage
- Query complexity

**When to use**: Document storage, high-volume writes, flexible schemas

### D. Data Lakes (Multi-tier Architecture)
**Examples**: Azure Data Lake, AWS Lake Formation, Databricks

**Architecture**: Bronze → Silver → Gold layers

**Pros**:
- Store all data types (structured, semi-structured, unstructured)
- Cost-effective at scale
- Supports advanced analytics and ML
- Separates storage and compute

**Cons**:
- Complexity in setup and governance
- Requires data catalog and metadata management
- Can become a "data swamp" without proper governance

**When to use**: Enterprise data analytics, multiple data sources, advanced ML/AI

## 3. Storage Cleanup and Lifecycle Management

### Why Delete Intermediate Data?

1. **Cost**: Storage isn't free. Intermediate files can accumulate quickly.
2. **Compliance**: GDPR, data retention policies require deleting old data.
3. **Performance**: Less data = faster queries and backups.
4. **Organization**: Avoid "data swamps" with too much clutter.

### Retention Policies

| Data Type | Retention Period | Reason |
|-----------|------------------|--------|
| **Raw Data** | 1-2 years | Legal compliance, reprocessing |
| **Cleaned Data** | 3-6 months | Can be regenerated from raw |
| **Intermediate ML Files** | 7-30 days | Temporary, debugging only |
| **Model Outputs** | 6-12 months | Auditing, A/B testing |
| **Production Models** | Until replaced | Active use |


This policy automatically:
- Moves raw data to Cool tier after 30 days
- Moves raw data to Archive tier after 180 days
- Deletes intermediate files after 7 days

## 4. Real-World Example: AH Dynamic Markdown Architecture

### What is Dynamic Markdown?
Albert Heijn's Dynamic Markdown system automatically adjusts product prices (markdowns/discounts) to reduce waste and optimize revenue for perishable goods.

### Data Storage Architecture

```
┌─────────────────────────────────────────────────────────────┐
│                  Data Sources (Extract)                     │
├─────────────────────────────────────────────────────────────┤
│  • POS Systems (Sales data)                                 │
│  • Inventory Management (Stock levels)                      │
│  • Weather APIs (Impact on demand)                          │
│  • Historical Pricing (Past markdowns)                      │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│         Azure Blob Storage (Bronze - Raw Data)              │
├─────────────────────────────────────────────────────────────┤
│  • Container: raw-data/                                     │
│    - sales_transactions/                                    │
│    - inventory_snapshots/                                   │
│    - weather_data/                                          │
│  • Retention: 2 years                                       │
│  • Tier: Hot → Cool (30 days) → Archive (180 days)         │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│         Azure Databricks (Transform/Process)                │
├─────────────────────────────────────────────────────────────┤
│  • Data cleaning and validation                             │
│  • Feature engineering (demand patterns, seasonality)       │
│  • ML model training (price optimization)                   │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│      Azure Blob Storage (Silver - Cleaned Data)             │
├─────────────────────────────────────────────────────────────┤
│  • Container: processed-data/                               │
│    - clean_sales/                                           │
│    - feature_store/                                         │
│  • Format: Parquet (optimized for analytics)                │
│  • Retention: 6 months                                      │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│          Azure ML / Model Registry (Models)                 │
├─────────────────────────────────────────────────────────────┤
│  • Trained markdown optimization models                     │
│  • Model versioning (A/B testing)                           │
│  • Model monitoring and drift detection                     │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│       Azure SQL Database (Gold - Model Outputs)             │
├─────────────────────────────────────────────────────────────┤
│  • Tables:                                                  │
│    - product_markdown_recommendations                       │
│    - markdown_history                                       │
│    - performance_metrics                                    │
│  • Indexed for fast queries                                 │
│  • Real-time access for store systems                       │
└─────────────────────────────────────────────────────────────┘
                            ↓
┌─────────────────────────────────────────────────────────────┐
│           PowerBI / Store Systems (Analytics)               │
├─────────────────────────────────────────────────────────────┤
│  • Store manager dashboards                                 │
│  • Category manager reports                                 │
│  • POS systems (apply markdowns)                            │
└─────────────────────────────────────────────────────────────┘
```

### Key Design Decisions

1. **Azure Blob Storage**: Scalable and cost-effective for large volumes of raw data
2. **Databricks**: Powerful for ETL and ML training at scale
3. **Parquet Format**: Columnar storage format for fast analytics (vs. CSV)
4. **SQL Database**: Fast real-time queries for operational decisions
5. **Lifecycle Policies**: Automatically move old data to cheaper storage

### Data Flow Example

1. **Morning (6 AM)**: POS systems upload yesterday's sales to Blob Storage
2. **Morning (7 AM)**: Databricks job processes data, trains/updates model
3. **Morning (8 AM)**: Model outputs markdown recommendations to SQL Database
4. **Morning (9 AM)**: Store managers review recommendations in PowerBI
5. **Throughout day**: POS systems query SQL Database for real-time pricing

## 5. Best Practices for Data Storage

### 1. Separate Storage from Compute
- Don't store data on the same server that runs computations
- Use cloud storage (Blob, S3) + separate compute (Databricks, Azure ML)
- Allows independent scaling

### 2. Use the Right Format
- **CSV**: Simple, human-readable, but slow and large
- **Parquet**: Columnar, compressed, fast for analytics (recommended)
- **JSON**: Flexible schema, good for APIs and NoSQL
- **Avro**: Schema evolution, good for streaming data

### 3. Partition Large Datasets
```
data/
  └── year=2024/
      └── month=12/
          └── day=23/
              └── product_reviews.parquet
```
Benefits:
- Faster queries (skip irrelevant partitions)
- Easier data management (delete old partitions)
- Parallel processing (process partitions in parallel)

### 4. Implement Data Versioning
- Track changes to datasets over time
- Useful for debugging and reproducibility
- Tools: DVC (Data Version Control), Azure ML datasets

### 5. Document Your Data
- Maintain a data catalog (Azure Purview, AWS Glue)
- Document schema, lineage, quality metrics
- Essential for collaboration and compliance

## Summary: Storage Theoretical Concepts

### Key Takeaways

1. **Data Lifecycle**: Raw → Cleaned → Model Outputs → Analytics
2. **Storage Options**: Local files, Cloud storage, Databases, Data lakes
3. **Cloud Storage**: Azure Blob Storage (Hot/Cool/Archive tiers)
4. **Lifecycle Management**: Automatically move or delete old data
5. **Real-World**: AH Dynamic Markdown uses Azure Blob + Databricks + SQL
6. **Best Practices**: Separate storage/compute, use Parquet, partition data

### How This Applies to Our Workshop

In this workshop:
- **Raw Data**: `data/raw/product_reviews.csv` (simulates Blob Storage bronze layer)
- **Cleaned Data**: `data/cleaned/cleaned_reviews.csv` (simulates silver layer)
- **Model Outputs**: Next notebook will generate predictions (simulates gold layer)

In production at AH:
- All these would be in Azure Blob Storage with proper lifecycle policies
- Data would be in Parquet format for better performance
- Final outputs would be in Azure SQL Database for real-time access

## Next Steps

In the next notebook (04_ml_analysis.ipynb), we'll:
- Train a sentiment analysis model on our cleaned data
- Make predictions on product reviews
- Create a discount recommendation system
- Demonstrate: "worst spinach ever" → 90% discount